In [ ]:
%load_ext autoreload
%autoreload 2
import os
import pandas as pd
import matplotlib.pyplot as plt

# Anchor to the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from src.config import SimConfig, EnvConfig
from src.utils.data_processing import load_and_cache_entire_fleet
from src.utils.evaluation import VoyageBenchmarker, print_markdown_table
from src.solvers import AugmentedHybridSDPSolver
from src.plants import AugmentedHybridPlant
from src.controllers import build_approach, AugmentedValueControl, AugmentedPolicyControl, AugmentedFCLockedControl

In [ ]:
# 1. Setup Environment and Load Fleet Data
env = EnvConfig()
fleet_data = load_and_cache_entire_fleet(env)

# Validation exclusion
exclude_days = [1, 2, 3]

In [ ]:
# 2. Base Setup for Iso-Complexity
seconds_in_day = 86400
divisors_86400 = sorted([d for d in range(1, seconds_in_day + 1) if seconds_in_day % d == 0])

dP_base = 150.0
Dt_base = 300.0
N_Pd = 6

def get_exact_complexity(dP, Dt, n_pack=4):
    """Calculates the exact number of Bellman node evaluations for a full 24h day."""
    cfg = SimConfig(
        dP=dP, Dt=Dt, N_Pd=N_Pd, use_smart_grid=True, n_pack=n_pack,
        apply_terminal_n_cost=False, apply_terminal_soc_cost=True, alpha_fc=4, verbose=False
    )
    # Extract actual lengths after the physics constraints snap the grid
    N_d = cfg.N_Pd
    N_n = len(cfg.n_vals)
    N_s = cfg.N_soc
    N_f = len(cfg.pfc_vals)
    N_b = len(cfg.pb_vals)
    
    S_nodes = N_d * N_n * N_s * N_f
    A_nodes = N_n * N_b
    steps = seconds_in_day / Dt
    
    # Total operations: T * |S| * (|N_d| + |A|)
    total_ops = steps * S_nodes * (N_d + A_nodes)
    return total_ops

# Baseline target complexity
target_complexity = get_exact_complexity(dP_base, Dt_base)

dp_test_range = range(50,255,10)

sweep_configurations = []
audit_data = []

print("--- RUNNING EXACT COMBINATORIAL SEARCH ---")
print(f"Target Operations: {target_complexity:,.0f} per day\n")

for dp in dp_test_range:
    # 1. The Pure Continuous Theory
    dt_theo = Dt_base * ((dP_base / dp) ** 1.5)
    
    # 2. Search for the Absolute Best Unconstrained Integer Dt
    search_min = max(1, int(dt_theo * 0.5))
    search_max = int(dt_theo * 2.0)
    best_dt_any = min(range(search_min, search_max), key=lambda dt: abs(get_exact_complexity(dp, dt) - target_complexity))
    ops_any = get_exact_complexity(dp, best_dt_any)
    error_any = ((ops_any - target_complexity) / target_complexity) * 100
    
    # 3. Search for the Best Clean Divisor Dt
    best_dt_div = min(divisors_86400, key=lambda dt: abs(get_exact_complexity(dp, dt) - target_complexity))
    ops_div = get_exact_complexity(dp, best_dt_div)
    error_div = ((ops_div - target_complexity) / target_complexity) * 100
    
    # --- CHOOSE YOUR APPROACH HERE ---
    # Switch between `best_dt_any` and `best_dt_div`
    # chosen_dt = best_dt_any 
    chosen_dt = best_dt_div
    
    # Save to the array that Cell 4 will use
    sweep_configurations.append({
        "dP": dp, 
        "Dt": chosen_dt, 
        "Math Error [%]": error_div if chosen_dt == best_dt_div else error_any
    })
    
    audit_data.append({
        'dP [kW]': dp,
        'Dt (Theory)': f"{dt_theo:.0f}",
        'Dt (Any Int)': best_dt_any,
        'Error (Any Int)': f"{error_any:+.1f}%",
        'Dt (Divisor)': best_dt_div,
        'Error (Divisor)': f"{error_div:+.1f}%",
    })

df_audit = pd.DataFrame(audit_data)
print_markdown_table(df_audit.set_index('dP [kW]'))

In [ ]:
# 3. Run the Benchmark Sweep
results_data = []

print("--- RUNNING ISO-COMPLEXITY SIMULATIONS ---")

for params in sweep_configurations:
    dp = params["dP"]
    dt = params["Dt"]
    math_error = params["Math Error [%]"]
    print(f"\n[ Evaluating Grid: dP = {dp:^5} kW | Dt = {dt:^4} s ]")
    
    # Configure Simulation
    config = SimConfig(
        dP=dp, Dt=dt, N_Pd=N_Pd, use_smart_grid=True, n_pack=4,
        apply_terminal_n_cost=False, apply_terminal_soc_cost=True, alpha_fc=4, verbose=True
    )
    
    benchmarker = VoyageBenchmarker(fleet_data, env, config, exclude_days)
    value_factory = build_approach(
        controller_cls=AugmentedValueControl, plant_cls=AugmentedHybridPlant, 
        solver_cls=AugmentedHybridSDPSolver, is_macro=False
    )
    
    # Run Leave-One-Out
    report = benchmarker.run_leave_one_out(value_factory)
    avg_metrics = report.summary.loc['Average']
    
    # Store metrics
    results_data.append({
        'dP [kW]': dp,
        'Dt [s]': dt,
        'Total Cost [$]': avg_metrics['Total Cost [$]'],
        'Math Error [%]': math_error,
        'Offline Compute Time [s]': avg_metrics['Offline Compute Time [s]']
    })

# Convert to DataFrame
df_iso = pd.DataFrame(results_data)

# Calculate Empirical Time Error relative to the baseline configuration (dP_base)
baseline_time = df_iso.loc[df_iso['dP [kW]'] == dP_base, 'Offline Compute Time [s]'].values[0]
df_iso['Time Error [%]'] = ((df_iso['Offline Compute Time [s]'] - baseline_time) / baseline_time) * 100

# Format the percentage columns nicely
df_iso_display = df_iso.copy()
df_iso_display['Math Error [%]'] = df_iso_display['Math Error [%]'].apply(lambda x: f"{x:+.1f}%")
df_iso_display['Time Error [%]'] = df_iso_display['Time Error [%]'].apply(lambda x: f"{x:+.1f}%")

print("\n--- MASTER ISO-COMPLEXITY SUMMARY TABLE ---")
print_markdown_table(df_iso_display.set_index(['dP [kW]', 'Dt [s]']))

In [ ]:
# 4. Plotting with Marker Sizes proportional to Measured Offline Compute Time
fig, ax = plt.subplots(figsize=(10, 7))

dp_vals = df_iso['dP [kW]'].values
costs = df_iso['Total Cost [$]'].values
times = df_iso['Offline Compute Time [s]'].values

# Scale marker sizes dynamically based on compute time for visual verification
# Min size 50, Max size 400
marker_sizes = 50 + ((times - times.min()) / (times.max() - times.min() + 1e-6)) * 350

scatter = ax.scatter(dp_vals, costs, s=marker_sizes, c=times, cmap='viridis', edgecolor='black', alpha=0.8, zorder=3)
ax.plot(dp_vals, costs, linestyle='--', color='gray', alpha=0.5, zorder=2)

for idx, row in df_iso.iterrows():
    ax.annotate(f"dP={int(row['dP [kW]'])}\nDt={int(row['Dt [s]'])}", 
                (row['dP [kW]'], row['Total Cost [$]']),
                xytext=(12, 0), textcoords='offset points',
                fontsize=9, fontweight='bold', color='black',
                va='center')

cbar = plt.colorbar(scatter)
cbar.set_label('Measured Offline Compute Time [s]', rotation=270, labelpad=15)

ax.set_title("True Iso-Complexity Curve: Spatial vs Temporal Tradeoff", fontsize=14, fontweight='bold')
ax.set_xlabel("Spatial Resolution Step ($dP$ in kW)", fontsize=12)
ax.set_ylabel("Average Total Cost [$]", fontsize=12)
ax.grid(True, linestyle='--', alpha=0.4, zorder=1)

os.makedirs('figures', exist_ok=True)
plt.savefig('figures/true_iso_complexity_curve.png', dpi=300, bbox_inches='tight')
plt.show()